In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains # <-- NEW IMPORT
import time
import csv

def scrape_zillow_data(url, limit=10, output_file="zillow_test_data.csv"):
    chrome_options = Options()
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        print("Navigating to Zillow...")
        driver.get(url)
        
        # --- NEW CAPTCHA HANDLING BLOCK ---
        print("Checking for 'Press and Hold' CAPTCHA...")
        try:
            # Wait up to 5 seconds to see if the CAPTCHA page loads
            # Note: '#px-captcha' is a common ID for PerimeterX, but you may need to update this selector!
            captcha_element = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '#px-captcha'))
            )
            
            print("CAPTCHA detected. Attempting to press and hold...")
            actions = ActionChains(driver)
            
            # Move mouse to the button, click, and hold it down
            actions.move_to_element(captcha_element).click_and_hold().perform()
            
            # Hold the click for 12 seconds (PerimeterX usually requires 10 seconds)
            time.sleep(12) 
            
            # Release the mouse button
            actions.release().perform()
            print("Released CAPTCHA button. Waiting for validation...")
            
            # Give the browser time to process the CAPTCHA success and load the real page
            time.sleep(5) 
            
        except Exception:
            # If no CAPTCHA element is found within 5 seconds, it will timeout and jump here
            print("No CAPTCHA found, or it timed out. Proceeding to scrape...")
        # --- END CAPTCHA HANDLING BLOCK ---


        # Wait for the property cards to load
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-test="property-card-price"]')))
        
        # Scroll down slightly to trigger lazy-loading
        driver.execute_script("window.scrollTo(0, 500);")
        time.sleep(2) 
        
        print(f"Extracting up to {limit} records...\n")
        
        prices = driver.find_elements(By.CSS_SELECTOR, '[data-test="property-card-price"]')
        addresses = driver.find_elements(By.CSS_SELECTOR, '[data-test="property-card-addr"]')
        details = driver.find_elements(By.CSS_SELECTOR, '[data-test="property-card-details"]')
        
        with open(output_file, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Price', 'Address', 'Details'])
            
            for i, (price, address, detail) in enumerate(zip(prices, addresses, details)):
                if i >= limit:
                    break
                
                clean_price = price.text
                clean_address = address.text
                clean_detail = detail.text.replace('bds', 'beds').replace('ba', 'baths').replace('\n', ' ')
                
                writer.writerow([clean_price, clean_address, clean_detail])
                print(f"Record {i + 1} saved: {clean_address}")
                
        print(f"\nSuccessfully saved {limit} records to {output_file}.")
                
    except Exception as e:
        print(f"An error occurred: {e}")
        
    finally:
        driver.quit()

zillow_url = "https://www.zillow.com/homes/for_sale/?searchQueryState=%7B%22pagination%22%3A%7B%7D%2C%22isMapVisible%22%3Atrue%2C%22mapBounds%22%3A%7B%22west%22%3A-108.80824613948747%2C%22east%22%3A-90.15346098323747%2C%22south%22%3A18.21110571454254%2C%22north%22%3A41.42832558652022%7D%2C%22mapZoom%22%3A6%2C%22filterState%22%3A%7B%22sort%22%3A%7B%22value%22%3A%22globalrelevanceex%22%7D%7D%2C%22isListVisible%22%3Atrue%7D"

scrape_zillow_data(zillow_url, limit=10)

Navigating to Zillow...
Checking for 'Press and Hold' CAPTCHA...
CAPTCHA detected. Attempting to press and hold...
Released CAPTCHA button. Waiting for validation...
An error occurred: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff75eca3045+14585]
	chromedriver!GetHandleVerifier [0x7ff75eca30b0+145f0]
	chromedriver!(No symbol) [0x7ff75ea0db7d]
	chromedriver!(No symbol) [0x7ff75ea673ae]
	chromedriver!(No symbol) [0x7ff75ea676bc]
	chromedriver!(No symbol) [0x7ff75eab7a57]
	chromedriver!(No symbol) [0x7ff75eab45af]
	chromedriver!(No symbol) [0x7ff75ea59a18]
	chromedriver!(No symbol) [0x7ff75ea5a8f3]
	chromedriver!GetHandleVerifier [0x7ff75efaac5d+31c19d]
	chromedriver!GetHandleVerifier [0x7ff75efa536d+3168ad]
	chromedriver!GetHandleVerifier [0x7ff75efc7d8a+3392ca]
	chromedriver!GetHandleVerifier [0x7ff75ecbf445+30985]
	chromedriver!GetHandleVerifier [0x7ff75ecc84dc+39a1c]
	chromedriver!GetHandleVerifier [0x7ff75ecac634+1db74]
	chromedriver!GetHandleVerifier [0x7ff75ecac7e6+1